# Cell 1 — Install Dependencies
Run this cell to install required libraries, then **restart the session**.

In [ ]:
!pip install -q --upgrade mediapipe==0.10.14 insightface==0.7.3 onnxruntime-gpu ultralytics>=8.3.100 opencv-python-headless httpx pyngrok nest-asyncio fastapi uvicorn python-multipart
print("Installation complete. Please restart the session before running Cell 2.")

## Cell 2 — Configuration & Imports
Define all constants and import necessary libraries.

In [1]:
import os, cv2, json, base64, time, zipfile, shutil
import warnings
import logging
# Suppress TF and Protobuf warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
logging.getLogger('absl').setLevel(logging.ERROR)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict, deque

# Import pipeline dependencies
from ultralytics import YOLO
from insightface.app import FaceAnalysis
import mediapipe as mp

# Import server dependencies
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, FileResponse
import uvicorn

from kaggle_secrets import UserSecretsClient

# --- CONFIGURATION ---
NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
FACES_DIR = "/kaggle/input/cbms-faces/"
CLASSIFIER_PATH = "/kaggle/input/datasets/gallimus/cbms-classifier/activity_classifier_v2.pt"

YOLO_CONF = 0.45
FACE_MATCH_THRESHOLD = 0.40
ACTIVITY_CONFIDENCE = 0.90

# FIX-3: Raised CONFIRMATION_VOTES (was 3 → 8) and ACTIVITY_SAMPLE_EVERY
# (was 8 → 15).  Old values let 3 noisy predictions in <1 s trigger an alert.
CONFIRMATION_VOTES   = 8
ACTIVITY_SAMPLE_EVERY = 15

# FIX-1: WATCH_WINDOW must equal N_FRAMES used in training (was 16, model
# trained on 32).  Feeding 16-frame tensors to a 32-frame model pushes every
# prediction out-of-distribution.
WATCH_WINDOW = 32

EVIDENCE_PRE   = 3
EVIDENCE_POST  = 7
EVIDENCE_TOTAL = EVIDENCE_PRE + EVIDENCE_POST
ALERT_UNKNOWN_FACES    = True
UNKNOWN_MIN_FACE_CROPS = 1
JPEG_QUALITY = 70

ACTIVITY_RULES = {
    "littering": {"score_delta": -15, "min_conf": 0.88},
    "helping":   {"score_delta": +10, "min_conf": 0.85},
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[DEBUG] Config loaded.")
print(f"[DEBUG] Device detected: {DEVICE}")
print(f"[DEBUG] WATCH_WINDOW={WATCH_WINDOW}  CONFIRMATION_VOTES={CONFIRMATION_VOTES}  ACTIVITY_SAMPLE_EVERY={ACTIVITY_SAMPLE_EVERY}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


E0000 00:00:1776315178.915911     144 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776315178.978442     144 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776315179.492128     144 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776315179.492157     144 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776315179.492160     144 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776315179.492163     144 computation_placer.cc:177] computation placer already registered. Please check linka

[DEBUG] Config loaded.
[DEBUG] Device detected: cuda


## Cell 3 — LSTM Model Architecture
Definition of the 2-layer LSTM activity classifier.

In [2]:
class PoseActivityClassifier(nn.Module):
    """
    Bidirectional 2-layer GRU with mean pooling.
    GRU-1   input_size  → hidden1   (bidirectional → output hidden1*2)
    GRU-2   hidden1*2   → hidden2   (bidirectional → output hidden2*2)
    Mean pooling across time dimension
    Dropout
    Linear  hidden2*2   → num_classes
    """
    def __init__(self, input_size, hidden1, hidden2, num_classes, dropout=0.0):
        super().__init__()
        self.gru1       = nn.GRU(input_size,  hidden1, batch_first=True, bidirectional=True)
        self.gru2       = nn.GRU(hidden1 * 2, hidden2, batch_first=True, bidirectional=True)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2 * 2, num_classes)

    def forward(self, x):
        out, _ = self.gru1(x)
        out, _ = self.gru2(out)
        out    = torch.mean(out, dim=1)   # mean pool, not last timestep
        return self.classifier(self.dropout(out))

print("[DEBUG] Bidirectional GRU architecture defined.")

[DEBUG] Bidirectional GRU architecture defined.


## Cell 4 — Model Initialization
Load YOLO, InsightFace, and LSTM weights.

In [3]:
print("Loading YOLOv8...")
YOLO_MODEL = YOLO("yolov8n.pt")
if torch.cuda.is_available():
    YOLO_MODEL.to("cuda")
    print("  YOLO -> GPU")

print("Loading GRU classifier weights...")
if not os.path.exists(CLASSIFIER_PATH):
    raise FileNotFoundError(f"Classifier not found at {CLASSIFIER_PATH}. Ensure the dataset is attached.")

ckpt        = torch.load(CLASSIFIER_PATH, map_location=DEVICE, weights_only=False)

# Assert checkpoint matches this architecture — catches accidental loading of old LSTM weights
assert ckpt.get("architecture") == "bidirectional_gru", (
    f"Checkpoint architecture is '{ckpt.get('architecture')}', expected 'bidirectional_gru'. "
    f"Wrong model file attached?"
)

CLASS_NAMES = ckpt["class_names"]
FEATURES    = ckpt["features"]
N_FRAMES    = ckpt["n_frames"]

CLASSIFIER = PoseActivityClassifier(
    FEATURES, ckpt["hidden1"], ckpt["hidden2"], len(CLASS_NAMES)
).to(DEVICE)
CLASSIFIER.load_state_dict(ckpt["model_state"])
CLASSIFIER.eval()
print(f"  GRU classifier loaded: {CLASS_NAMES}  (val_acc={ckpt.get('val_accuracy', '?'):.1%})")

print("Loading InsightFace...")
FACE_APP = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
FACE_APP.prepare(ctx_id=0, det_size=(640, 640))
print("  InsightFace ready")

bytetrack_config = """
tracker_type: bytetrack
track_high_thresh: 0.35
track_low_thresh: 0.05
new_track_thresh: 0.35
track_buffer: 60
match_thresh: 0.70
fuse_score: True
"""
with open("/kaggle/working/bytetrack_custom.yaml", "w") as f:
    f.write(bytetrack_config)

print("\n[DEBUG] All models loaded.")

Loading YOLOv8...
  YOLO -> GPU
Loading GRU classifier weights...
  GRU classifier loaded: ['littering', 'normal', 'spitting']  (val_acc=94.8%)
Loading InsightFace...
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 103216.04KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

## Cell 5 — Face Enrollment
Enrolling faces from the specified directory.

In [4]:
def load_face_db(faces_dir):
    db = {}
    if not os.path.exists(faces_dir):
        print(f"WARNING: {faces_dir} not found")
        return db
    
    files = sorted([f for f in os.listdir(faces_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
    print(f"Enrolling {len(files)} persons...")
    
    for filename in files:
        name = os.path.splitext(filename)[0]
        img = cv2.imread(os.path.join(faces_dir, filename))
        if img is None: continue
        
        faces = FACE_APP.get(img)
        if faces:
            best = max(faces, key=lambda f: f.det_score)
            db[name] = best.embedding.astype(np.float32)
            print(f"  Enrolled: {name:<20} score={best.det_score:.3f}")
    
    print(f"\nEnrollment complete. Total persons: {len(db)}")
    return db

FACE_DB = load_face_db(FACES_DIR)

## Cell 6 — Stateful Pipeline Core
This class maintains the state of trackers and buffers across sequential chunks. Modifies the chunk to be returned as processed video.

In [5]:
class StatefulPipeline:
    def __init__(self):
        self.frame_idx    = 0
        self.pre_buffers  = defaultdict(lambda: deque(maxlen=EVIDENCE_PRE))
        self.kp_buffers   = defaultdict(lambda: deque(maxlen=WATCH_WINDOW))
        self.capture_state  = {}
        self.person_scores  = defaultdict(lambda: 100)
        self.vote_counters  = defaultdict(lambda: {"activity": "", "count": 0})

        # FIX-4: static_image_mode=True matches training extraction exactly.
        # The training notebook uses a single global Pose with static_image_mode=True,
        # meaning every frame is processed independently with no temporal carry-over.
        # Using False here creates a detection-quality mismatch between training and inference.
        self.pose_trackers   = {}
        self.track_last_seen = {}

    # ── FIX-2: run pose on the FULL FRAME, not on the bounding-box crop ───────
    # Training extracts landmarks from full frames; the person occupies a small
    # fraction so landmark coords are frame-normalised.  When we resized the
    # 64×64 crop instead, the person filled the entire image, so all landmark
    # coords were at a completely different scale even after nose-centering.
    #
    # New signature: pass the full frame + the tight YOLO bbox.
    # We run MediaPipe on the full frame and only accept the result when the
    # detected nose landmark falls inside the tracked bounding box.
    # This guarantees we're tracking the right person in multi-person scenes.
    def extract_keypoints(self, tid, frame, x1, y1, x2, y2):
        if frame is None or frame.size == 0:
            return np.zeros(FEATURES, dtype=np.float32)

        if tid not in self.pose_trackers:
            print(f"[DEBUG] Starting new MediaPipe instance for Track ID: {tid}")
            # FIX-4: static_image_mode=True — matches training
            self.pose_trackers[tid] = mp.solutions.pose.Pose(
                static_image_mode=True,
                min_detection_confidence=0.3,
                min_tracking_confidence=0.3)

        try:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = self.pose_trackers[tid].process(rgb)
            if res.pose_landmarks:
                kp   = res.pose_landmarks.landmark
                nose = kp[0]

                # Validate: nose must be inside the tracked person's bbox.
                # MediaPipe coords are normalised [0,1] relative to full frame.
                h_f, w_f = frame.shape[:2]
                nose_px_x = nose.x * w_f
                nose_px_y = nose.y * h_f
                # Allow 20 % margin around the bbox to handle detection jitter.
                margin_x = (x2 - x1) * 0.20
                margin_y = (y2 - y1) * 0.20
                if not (x1 - margin_x <= nose_px_x <= x2 + margin_x and
                        y1 - margin_y <= nose_px_y <= y2 + margin_y):
                    return np.zeros(FEATURES, dtype=np.float32)

                return np.array(
                    [[lm.x - nose.x, lm.y - nose.y, lm.z] for lm in kp],
                    dtype=np.float32
                ).flatten()
        except Exception as e:
            print(f"[WARN] Keypoint extraction error (TID {tid}): {e}")
        return np.zeros(FEATURES, dtype=np.float32)

    def classify_keypoints(self, kp_buffer):
        if len(kp_buffer) < WATCH_WINDOW: return "normal", 0.0
        seq = np.stack(list(kp_buffer)[-WATCH_WINDOW:])
        x   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(CLASSIFIER(x), dim=1).cpu().numpy()[0]
        best_idx  = int(probs.argmax())
        best_name = CLASS_NAMES[best_idx]
        return (best_name, float(probs[best_idx])) if best_name != "normal" else ("normal", 0.0)

    def identify_from_crops(self, crops):
        scores_acc = defaultdict(float)
        valid, face_count = 0, 0
        indices = np.linspace(0, len(crops)-1, 4, dtype=int)
        subset  = [crops[i] for i in indices]
        for crop in subset:
            if crop is None or crop.size == 0: continue
            try:
                faces = FACE_APP.get(crop)
                if not faces: continue
                best = max(faces, key=lambda f: f.det_score)
                if best.det_score < 0.50: continue
                face_count += 1
                if not FACE_DB: continue
                valid += 1
                for name, ref in FACE_DB.items():
                    sim = float(np.dot(best.embedding / np.linalg.norm(best.embedding),
                                       ref / np.linalg.norm(ref)))
                    if sim > FACE_MATCH_THRESHOLD: scores_acc[name] += sim
            except: pass
        if not scores_acc or valid == 0: return None, 0.0, face_count
        best_name = max(scores_acc, key=scores_acc.get)
        return best_name, round(scores_acc[best_name] / valid, 3), face_count

    def _make_evidence_grid(self, crops):
        blank  = np.zeros((128, 128, 3), dtype=np.uint8)
        thumbs = [cv2.resize(c, (128, 128)) if c is not None and c.size > 0 else blank for c in crops]
        while len(thumbs) < EVIDENCE_TOTAL: thumbs.append(blank)
        grid = np.vstack([np.hstack(thumbs[:5]), np.hstack(thumbs[5:10])])
        _, buf = cv2.imencode(".jpg", grid, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        return base64.b64encode(buf).decode()

    def process_chunk(self, video_path):
        cap    = cv2.VideoCapture(video_path)
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps    = cap.get(cv2.CAP_PROP_FPS) or 15.0

        output_path = "/kaggle/working/processed_" + os.path.basename(video_path)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        alerts_generated = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            h_f, w_f = frame.shape[:2]

            try:
                res = YOLO_MODEL.track(
                    frame, conf=YOLO_CONF, classes=[0], persist=True,
                    tracker="/kaggle/working/bytetrack_custom.yaml", verbose=False)[0]
            except Exception as e:
                print(f"[ERROR] YOLO Tracking Failed: {e}")
                out.write(frame)
                continue

            active_tids = set()
            if res.boxes is not None and res.boxes.id is not None:
                for box in res.boxes:
                    try:
                        tid = int(box.id[0])
                        active_tids.add(tid)
                        self.track_last_seen[tid] = self.frame_idx
                        x1 = max(0,   int(box.xyxy[0][0]))
                        y1 = max(0,   int(box.xyxy[0][1]))
                        x2 = min(w_f, int(box.xyxy[0][2]))
                        y2 = min(h_f, int(box.xyxy[0][3]))
                        if x2 <= x1 or y2 <= y1: continue
                        crop = frame[y1:y2, x1:x2].copy()
                        self.pre_buffers[tid].append(crop)

                        if tid in self.capture_state:
                            cs = self.capture_state[tid]
                            cs["frames"].append(crop)
                            if len(cs["frames"]) >= EVIDENCE_TOTAL:
                                act  = cs["activity"]
                                rule = ACTIVITY_RULES.get(act, {})
                                name, id_conf, face_count = self.identify_from_crops(cs["frames"])
                                grid = self._make_evidence_grid(cs["frames"])
                                if name and id_conf >= rule.get("min_conf", 1.0):
                                    self.person_scores[name] += rule.get("score_delta", 0)
                                    alerts_generated.append({
                                        "person_name": name, "identity": "known",
                                        "activity": act, "activity_conf": float(cs["activity_conf"]),
                                        "id_confidence": id_conf,
                                        "new_score": self.person_scores[name],
                                        "evidence_grid_b64": grid})
                                    print(f"[ALERT] Triggered for {name}: {act}")
                                elif ALERT_UNKNOWN_FACES and face_count >= UNKNOWN_MIN_FACE_CROPS:
                                    alerts_generated.append({
                                        "person_name": f"UNKNOWN-{tid}", "identity": "unknown",
                                        "activity": act, "activity_conf": float(cs["activity_conf"]),
                                        "id_confidence": 0.0, "new_score": 0,
                                        "evidence_grid_b64": grid})
                                    print(f"[ALERT] Triggered for Unknown-{tid}: {act}")
                                del self.capture_state[tid]
                        else:
                            # FIX-2: pass full frame + bbox coords instead of crop
                            kp = self.extract_keypoints(tid, frame, x1, y1, x2, y2)
                            self.kp_buffers[tid].append(kp)
                            if (self.frame_idx % ACTIVITY_SAMPLE_EVERY == 0 and
                                    len(self.kp_buffers[tid]) == WATCH_WINDOW):
                                act, conf = self.classify_keypoints(self.kp_buffers[tid])
                                if act != "normal" and conf >= ACTIVITY_CONFIDENCE:
                                    cv = self.vote_counters[tid]
                                    cv["count"] = cv["count"] + 1 if cv["activity"] == act else 1
                                    cv["activity"] = act
                                    if cv["count"] >= CONFIRMATION_VOTES:
                                        self.vote_counters[tid] = {"activity": "", "count": 0}
                                        self.capture_state[tid] = {
                                            "activity": act, "activity_conf": float(conf),
                                            "frames": list(self.pre_buffers[tid])}
                                        print(f"[DEBUG] Triggered capture for {act} TID {tid}")

                        capturing = tid in self.capture_state
                        color = (0, 140, 255) if capturing else (0, 210, 90)
                        label = f"ID:{tid}"
                        if capturing:
                            label += f" {self.capture_state[tid]['activity']}"
                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                        cv2.putText(frame, label, (x1, max(y1-8, 12)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
                    except Exception as e:
                        print(f"[ERROR] Box processing error TID {box.id[0]}: {e}")
            else:
                if self.frame_idx % 30 == 0:
                    print("[DEBUG] res.boxes.id is None (No active tracks)")

            self.frame_idx += 1
            tids_to_del = [t for t, last in self.track_last_seen.items()
                           if (self.frame_idx - last) > 60]
            for t in tids_to_del:
                if t in self.pose_trackers:
                    print(f"[DEBUG] Closing MediaPipe instance for lost track TID: {t}")
                    self.pose_trackers[t].close()
                    del self.pose_trackers[t]
                del self.track_last_seen[t]
            out.write(frame)

        cap.release()
        out.release()
        return alerts_generated, output_path

PIPELINE = StatefulPipeline()
print("[DEBUG] Stateful Pipeline instantiated.")
print(f"[DEBUG] Keypoint extraction: full-frame mode (matches training).")
print(f"[DEBUG] MediaPipe: static_image_mode=True (matches training).")


[DEBUG] Stateful Pipeline class instantiated.


## Cell 7 — API Layer (FastAPI)
Endpoints for processing chunks and retrieving scores.

In [6]:
import asyncio
import nest_asyncio
import threading
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse, FileResponse
import shutil
import os
import json
import re
import time

app = FastAPI()

# Sequential processing lock
process_lock = threading.Lock()
process_cond = threading.Condition(process_lock)
expected_chunk_idx = 0

@app.post("/process_chunk")
async def process_chunk(file: UploadFile = File(...)):
    global expected_chunk_idx
    
    # Extract chunk index from filename (chunk_0001.mp4)
    m = re.search(r'chunk_(\d+)', file.filename)
    chunk_idx = int(m.group(1)) if m else -1

    temp_path = f"/kaggle/working/temp_{file.filename}"
    with open(temp_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    
    try:
        # Wait for our turn to process sequentially (Client handles timeouts and resets now)
        with process_cond:
            if chunk_idx != -1:
                while chunk_idx != expected_chunk_idx:
                    process_cond.wait(timeout=1.0)
            
            print(f"Processing chunk: {file.filename} (IDX: {chunk_idx}, Expected: {expected_chunk_idx})")
            pt_start = time.time()
            alerts, output_path = PIPELINE.process_chunk(temp_path)
            pt_elapsed = time.time() - pt_start
            print(f"[✓] Done in {pt_elapsed:.2f} seconds.")
            
            if chunk_idx != -1:
                # Update expected next chunk
                expected_chunk_idx = chunk_idx + 1
                process_cond.notify_all()
        
        alerts_for_header = []
        for a in alerts:
            a_cp = dict(a)
            if "evidence_grid_b64" in a_cp:
                del a_cp["evidence_grid_b64"]
            alerts_for_header.append(a_cp)
        
        headers = {
            "X-Global-Frame": str(PIPELINE.frame_idx),
            "X-Alerts": json.dumps(alerts_for_header),
            "X-Chunk-Idx": str(chunk_idx)
        }
        return FileResponse(path=output_path, headers=headers, media_type="video/mp4")
    
    except Exception as e:
        print(f"[ERROR] Process chunk failed for {file.filename}: {e}")
        return JSONResponse({"error": str(e)}, status_code=500)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

@app.get("/scores")
def get_scores():
    return JSONResponse(content=dict(PIPELINE.person_scores))

@app.get("/health")
def health():
    return {
        "status": "ok",
        "global_frame": PIPELINE.frame_idx,
        "active_trackers": len(PIPELINE.pose_trackers),
        "enrolled": len(FACE_DB),
        "expected_chunk": expected_chunk_idx
    }

@app.post("/reset_pipeline")
def reset_pipeline():
    global PIPELINE, expected_chunk_idx
    with process_cond:
        # Re-instantiate the pipeline object
        PIPELINE = StatefulPipeline()
        expected_chunk_idx = 0
        process_cond.notify_all()
    print("[RESET] Pipeline and sequential counter reset.")
    return {"status": "reset"}

@app.post("/set_expected_chunk/{idx}")
def set_expected_chunk(idx: int):
    global expected_chunk_idx
    with process_cond:
        print(f"[RESYNC] Server told to discard old queue and expect chunk_{idx:04d} next.")
        expected_chunk_idx = idx
        process_cond.notify_all()
    return {"status": "resynced", "expected": expected_chunk_idx}

print("[DEBUG] FastAPI App routes defined with sequential processing lock & resync.")

[DEBUG] FastAPI App routes defined with sequential processing lock & resync.


## Cell 8 — Server Launch
Start the Ngrok tunnel and Uvicorn server.

In [ ]:
import asyncio
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# Apply nest_asyncio FIRST — allows asyncio in Jupyter
nest_asyncio.apply()

# Start ngrok tunnel
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any existing tunnels to avoid duplicates
ngrok.kill()
tunnel     = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "="*60)
print(f"API LIVE at: {public_url}")
print(f"Send .mp4 chunks to: {public_url}/process_chunk")
print(f"Check scores at    : {public_url}/scores")
print(f"Health check       : {public_url}/health")
print("="*60 + "\n")

# Run uvicorn using run_until_complete — works inside Kaggle cells
# nest_asyncio.apply() above makes this safe in an existing event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000,
                         loop="asyncio", log_level="warning")
server = uvicorn.Server(config)

# This blocks the cell and keeps the server alive
# Interrupt the kernel (square stop button) to shut down
print("Server running... interrupt kernel to stop.")
asyncio.get_event_loop().run_until_complete(server.serve())


                                                                                                    
API LIVE at: https://malapportioned-synostotic-freeda.ngrok-free.dev
Send .mp4 chunks to: https://malapportioned-synostotic-freeda.ngrok-free.dev/process_chunk
Check scores at    : https://malapportioned-synostotic-freeda.ngrok-free.dev/scores
Health check       : https://malapportioned-synostotic-freeda.ngrok-free.dev/health

Server running... interrupt kernel to stop.
[RESET] Pipeline and sequential counter reset.
[RESYNC] Server told to discard old queue and expect chunk_0000 next.
Processing chunk: chunk_0000.mp4 (IDX: 0, Expected: 0)
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 239ms
Prepared 1 package in 86ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effec

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776315373.780264     283 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776315373.818499     283 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[DEBUG] res.boxes.id is None (No active tracks)
[✓] Done in 3.81 seconds.
Processing chunk: chunk_0004.mp4 (IDX: 4, Expected: 4)
[✓] Done in 5.97 seconds.
[RESYNC] Server told to discard old queue and expect chunk_0005 next.
Processing chunk: chunk_0005.mp4 (IDX: 5, Expected: 5)
[✓] Done in 6.00 seconds.
